# DoorDash Delivery ETA — Etapa 04: Feature Engineering

Este notebook executa e audita o módulo `src/feature_engineering.py` sobre `df_model` (`197.421` registros válidos), verificando:
1. Investigação de fuso horário (`created_at`) antes de definir períodos do dia.
2. Criação das features de pedido, operacionais, estimativa experimental e contagem histórica estritamente causal por loja.
3. Ausências, distribuições e controle rigoroso contra *data leakage*.
4. Separação entre features com **disponibilidade confirmada** e **disponibilidade incerta/experimental**.

> **Regras desta etapa:** Nenhum modelo é treinado e nenhuma variável derivada do alvo (`actual_delivery_time`, `delivery_duration_*`) é incluída nos conjuntos de features.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preparation import prepare_datasets
from src.feature_engineering import (
    build_features,
    CONFIRMED_FEATURES,
    UNCERTAIN_EXPERIMENTAL_FEATURES,
    FORBIDDEN_LEAKAGE_COLS,
    FEATURE_AVAILABILITY,
)

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "historical_data.csv"
_, _, df_model, _ = prepare_datasets(RAW_PATH)
df_feat = build_features(df_model)

print(f"Shape após Feature Engineering: {df_feat.shape}")
print(f"Features confirmadas: {len(CONFIRMED_FEATURES)}")
print(f"Features incertas/experimentais: {len(UNCERTAIN_EXPERIMENTAL_FEATURES)}")

## 1. Verificação de Fuso Horário (`created_at`)

In [ ]:
zero_hours = sorted(set(range(24)) - set(df_feat["created_hour"].unique()))
print("Horas sem nenhum pedido registrado:", zero_hours)
display(df_feat.groupby("market_id", observed=False)["created_hour"].value_counts().groupby(level=0).head(3).to_frame("pedidos"))

## 2. Auditoria das Novas Features (Ausências e Distribuições)

In [ ]:
new_cols = [
    "created_hour", "created_dayofweek", "created_is_weekend", "created_time_block",
    "average_item_price", "item_price_range", "distinct_item_ratio", "is_single_item", "is_large_order",
    "dasher_count_difference", "busy_greater_than_onshift", "is_operational_missing",
    "outstanding_orders_per_onshift_dasher", "busy_dashers_ratio",
    "estimated_total_duration", "historical_store_order_count"
]

rows = []
for col in new_cols:
    na_cnt = int(df_feat[col].isna().sum())
    na_pct = round(na_cnt / len(df_feat) * 100, 2)
    if df_feat[col].dtype.name in ["category", "boolean"]:
        rows.append({"feature": col, "nulls": na_cnt, "null_%": na_pct, "min": None, "p50": None, "mean": None, "p95": None, "max": None})
    else:
        s = df_feat[col].dropna()
        rows.append({
            "feature": col, "nulls": na_cnt, "null_%": na_pct,
            "min": round(s.min(), 2), "p50": round(s.median(), 2),
            "mean": round(s.mean(), 2), "p95": round(s.quantile(0.95), 2), "max": round(s.max(), 2),
        })
display(pd.DataFrame(rows))

## 3. Controle de Vazamento e Classificação de Disponibilidade

In [ ]:
overlap_confirmed = set(CONFIRMED_FEATURES) & set(FORBIDDEN_LEAKAGE_COLS)
overlap_uncertain = set(UNCERTAIN_EXPERIMENTAL_FEATURES) & set(FORBIDDEN_LEAKAGE_COLS)
assert len(overlap_confirmed) == 0 and len(overlap_uncertain) == 0, "Vazamento detectado!"
print("Verificação anti-leakage aprovada: nenhuma coluna proibida nos conjuntos de features.")

avail_df = pd.DataFrame.from_dict(FEATURE_AVAILABILITY, orient="index").reset_index()
avail_df.columns = ["feature", "disponibilidade", "risco_observacao"]
display(avail_df)